In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from churn import config
from churn.features import build_pipeline


In [2]:
df=pd.read_parquet(config.CLEAN_DATA_FILE)
X = df.drop(columns=[config.TARGET])
y = df[config.TARGET]

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=config.RANDOM_STATE, stratify=y)
print(f"X_train.shape: {X_train.shape}, X_test.shape: {X_test.shape}")
print(f"churn rate  train={y_train.mean():.4f}  test={y_test.mean():.4f}")

X_train.shape: (5634, 20), X_test.shape: (1409, 20)
churn rate  train=0.2654  test=0.2654


In [4]:
from sklearn.linear_model import LogisticRegression
from churn.evaluate import run_cv

In [5]:
pipe_lr = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr")

#### Model Comparison DataFrame

In [6]:
## This list will hold the results of cross-validation for different models. Each entry will be a dictionary containing the model name and its performance metrics.
results_cv = [run_cv(pipe_lr, "Logistic Regression", X=X_train, y=y_train)]
model_compare_df = pd.DataFrame(results_cv)
model_compare_df 

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.01299,0.659564,0.018448,0.654277,0.53913,0.590424,0.003014,0.12504


## Step 3: KNN and SVM: why they may struggle on this data

### KNN

**How KNN measures "similar":** it measures the distance between two customers across all model columns (Euclidean by default, which is Minkowski with p=2; Manhattan is p=1). The *k* customers with the smallest distance are the "neighbours", and their churn rate becomes the prediction.

**Why many 0/1 one-hot columns make this harder:**
1. **The distance can only take a few values.** On a one-hot column two customers either match (difference 0) or don't (difference 1). Across ~20 such columns, the distance mostly just counts *how many categories differ*, so many customers end up at exactly the same distance. Ties are common, and "nearest" becomes fuzzy.
2. **Every column counts equally.** KNN doesn't learn which columns matter. A mismatch on `Contract` (the strongest churn driver) adds the same distance as a mismatch on `StreamingTV` (almost no signal). LR, by contrast, learns a weight per column.
3. **One-hot inflates some features.** A 3-level feature such as `Contract` becomes 3 columns, so a mismatch there adds 2 to the squared distance, while a yes/no feature adds only 1. Feature weight then depends on the *number of levels*, not on importance.
4. **Scaled numerics vs 0/1 columns.** `tenure` and `MonthlyCharges` are standardised (spread about −2 to +2). The two numeric columns compete with ~20 binary ones, so how much tenure matters is a side effect of scaling, not something learned.
5. **Curse of dimensionality.** With more columns, the nearest and the farthest customers become more alike in distance, so "near" carries less information.

**SVM** also uses distances (the RBF kernel), so points 2–5 partly apply. In addition, it doesn't output probabilities natively: `probability=True` adds an internal calibration step (extra CV), which makes it slower.

**Expectation:** neither should beat LR here. LR learns one weight per column, and the churn signal in this data is mostly additive (contract, tenure, fiber, payment method).

In [7]:
from sklearn.neighbors import KNeighborsClassifier

In [8]:
pipe_knn = build_pipeline(KNeighborsClassifier(n_neighbors=5), model_type="lr")

In [9]:
results_cv.append(run_cv(pipe_knn, "KNN", X=X_train, y=y_train))
model_compare_df = pd.DataFrame(results_cv)
model_compare_df 

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.125040
1,KNN,0.779182,0.018299,0.521791,0.032812,0.577296,0.520401,0.547279,0.121479,0.109721


#### KNN (k=5): predictions vs result

| | LR | KNN (k=5) |
|---|---|---|
| CV ROC-AUC | 0.845 | **0.779** |
| CV PR-AUC | 0.660 | **0.522** |
| gap (train − CV ROC-AUC) | 0.003 | **0.121** |
| fit_time (s) | 0.16 | 0.10 |

**Q1. Will KNN's ROC-AUC be higher or lower than LR's 0.845?**
- *My prediction:* lower, because of the many binary columns and because most neighbours are non-churners.
- *Result:* lower (0.779) ✅
- *Corrected reason:* the class imbalance mainly affects the 0.5 cut-off (precision/recall). ROC-AUC uses no cut-off; it only asks whether churners get higher scores than non-churners. KNN ranks worse because **its distance is a poor measure of "similar" here**: every column counts equally (a `Contract` mismatch weighs the same as a `StreamingTV` mismatch), one-hot columns create many ties, and the model learns no weights. LR learns one weight per column.

**Q2. Will KNN's train−CV gap be bigger or smaller than LR's 0.003?**
- *My prediction:* smaller.
- *Result:* much bigger (0.121, about 40× LR) ❌
- *Corrected reason:* to get the train score, KNN scores customers it has already stored. Each one's nearest neighbour is **itself** (distance 0), so 1 of the 5 votes is its own true label. That's memorisation, which inflates the train score; CV folds have no self-vote. **Small k = overfitting** (k=1 gives train ROC-AUC ≈ 1.0); larger k averages more customers and smooths predictions. The gap fails L6 (< 0.05), so `n_neighbors` gets tuned in step 9.

**Fit time:** KNN's "fit" only stores the data (a *lazy learner*); the neighbour search happens at prediction time. Scoring a large customer base would be slow, a practical point against deploying it.

**Takeaway:** untuned KNN is clearly worse than LR at ranking (−0.066 ROC-AUC) and overfits. It stays in the table as a comparison point, not a finalist candidate.

### SVM

In [10]:
from sklearn.svm import SVC
pipe_svm = build_pipeline(SVC(probability=True, random_state=config.RANDOM_STATE), model_type="lr")
results_cv.append(run_cv(pipe_svm, "SVM (RBF)", X=X_train, y=y_train))

In [11]:
model_compare_df = pd.DataFrame(results_cv)
model_compare_df 

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.125040
1,KNN,0.779182,0.018299,0.521791,0.032812,0.577296,0.520401,0.547279,0.121479,0.109721
2,SVM (RBF),0.795707,0.017552,0.636447,0.021620,0.656116,0.503679,0.569373,0.070347,2.263394


#### SVM (RBF): predictions vs result

| | LR | KNN (k=5) | SVM (RBF) |
|---|---|---|---|
| CV ROC-AUC | 0.845 | 0.779 | **0.796** |
| CV PR-AUC | 0.660 | 0.522 | **0.636** |
| precision / recall | 0.654 / 0.539 | 0.577 / 0.520 | **0.656 / 0.504** |
| gap (train − CV ROC-AUC) | 0.003 | 0.121 | **0.070** |
| fit_time (s) | 0.13 | 0.13 | **3.11** |

**Q1. Will SVM's ROC-AUC be below KNN, between KNN and LR, or above LR?**
- *My prediction:* between KNN and LR.
- *Result:* between (0.796) ✅
- *Corrected reason:* LR also draws a boundary (its probability is a sigmoid of the distance from a *straight* boundary), so "boundary vs probability" isn't the difference. The useful question is **what each model learns**:

| Model | What it learns | How it uses distance |
|---|---|---|
| LR | one weight **per column** | doesn't use customer-to-customer distance |
| SVM (RBF) | one weight **per training customer** (only the support vectors count) | similarity = `exp(−gamma × distance²)` |
| KNN | nothing, it just stores the data | raw distance, then a vote of the 5 nearest |

SVM beats KNN because it learns *which* training customers matter and by how much, with a smooth margin and regularisation (`C`). It stays below LR because its RBF similarity is built on the same distance as KNN (every column counts equally, one-hot inflation), and the churn signal here is mostly additive, so a curved boundary gains little over a straight one.

**Q2. Will SVM's train−CV gap be bigger or smaller than KNN's 0.121? Under 0.05?**
- *My prediction:* similar to KNN, above 0.05.
- *Result:* above 0.05 ✅ but clearly smaller than KNN (0.070 vs 0.121) ❌
- *Corrected reason:* SVM doesn't memorise every customer. It keeps only the support vectors, and the soft margin (`C`) allows some training mistakes on purpose. There is no self-vote. The RBF boundary can still bend around individual training points, which gives the 0.07. It fails L6, so `C` and `gamma` get tuned in step 9 (lower values give a smoother boundary).

**Q3. Fit time vs LR and KNN?**
- *My prediction:* slower, because it has to find the closest points and choose the kernel.
- *Result:* slower, about 23× LR ✅
- *Corrected reason:* SVM doesn't choose the kernel (we fixed it to RBF). There are two real causes: (1) kernel SVM compares *pairs* of customers, so its cost grows roughly with n² to n³ rows (LR grows roughly linearly); (2) `probability=True` runs an extra internal 5-fold CV to calibrate the probabilities, so each CV fold does about 6 fits instead of 1. SVM doesn't scale to millions of customers.

**Q4. Why does SVC need `probability=True`?**
- *My prediction:* didn't know.
- *Answer:* ROC-AUC and PR-AUC only need a **ranking**, and SVM's `decision_function` (a signed distance from the boundary) gives that. Real probabilities are needed in **phase 6**: the cost-based threshold ("flag if P(churn) × lost revenue > offer cost") and the app's "70% churn risk" only make sense with real probabilities. A distance isn't a probability, so Platt scaling fits a sigmoid on top of it, using that extra internal CV.

**Q5. Compared with LR, will SVM flag more churners or fewer at the default cut-off?**
- *My prediction:* didn't know.
- *Answer:* fewer, with recall 0.504 vs 0.539 and about the same precision. With no class weights every mistake costs the same, so the 73% non-churners push the boundary toward the churner side and only the clearer churners cross it. This affects the **cut-off, not the ranking**. It gets fixed with `class_weight` (step 8) or the threshold (phase 6).

**Observation:** SVM's PR-AUC (0.636) is close to LR's (0.660), but its ROC-AUC is far behind. PR-AUC mostly rewards the **top** of the ranking, while ROC-AUC covers all of it. So SVM probably ranks its most confident churners well and the uncertain middle poorly. For a retention team that only calls the top few hundred customers, SVM is less weak than its ROC-AUC suggests, but it's still below LR on both metrics.

**Takeaway:** the default SVM is better than KNN, below LR, ~23× slower, and overfits (gap 0.07, fails L6). It stays in the table as a comparison point, not a finalist candidate.

## Step 4: Decision tree

In [12]:
from sklearn.tree import DecisionTreeClassifier
pipe_dt = build_pipeline(DecisionTreeClassifier(random_state=config.RANDOM_STATE), model_type="tree")
results_cv.append(run_cv(pipe_dt, "Decision Tree (default)", X=X_train, y=y_train))

In [13]:
model_compare_df = pd.DataFrame(results_cv)
model_compare_df 

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.125040
1,KNN,0.779182,0.018299,0.521791,0.032812,0.577296,0.520401,0.547279,0.121479,0.109721
2,SVM (RBF),0.795707,0.017552,0.636447,0.021620,0.656116,0.503679,0.569373,0.070347,2.263394
3,Decision Tree (default),0.654996,0.015929,0.376298,0.016837,0.483949,0.500334,0.491915,0.344985,0.192541


#### Decision Tree (default): predictions vs result

| | LR | KNN (k=5) | SVM (RBF) | Tree (default) |
|---|---|---|---|---|
| CV ROC-AUC | 0.845 | 0.779 | 0.796 | **0.655** |
| CV PR-AUC | 0.660 | 0.522 | 0.636 | **0.376** |
| gap (train − CV ROC-AUC) | 0.003 | 0.121 | 0.070 | **0.345** |
| fit_time (s) | 0.13 | 0.11 | 2.26 | **0.19** |

**Q1. Will the default tree's train ROC-AUC be close to 1.0? Will its gap be bigger or smaller than KNN's 0.121?**
- *My prediction:* train ROC-AUC not close to 1.0; gap smaller than KNN, because the tree learns the underlying pattern by splitting (reducing entropy) rather than looking only at the nearest neighbours.
- *Result:* train ROC-AUC = 0.655 + 0.345 = **1.000**, and the gap is about 3× KNN's ❌ ❌
- *Corrected reason:* with no `max_depth` and `min_samples_leaf=1`, the tree **keeps splitting until every leaf is pure**, often one customer per leaf. Only the **top splits** (Contract, tenure) learn real patterns. The deep splits learn **noise** ("this one customer churned"). That is complete memorisation, worse than KNN, which at least averages 5 neighbours.
- *Why the CV ROC-AUC is so low:* every leaf is pure, so every predicted probability is exactly **0 or 1**. The tree can't rank customers, only make a yes/no call, so the ROC curve is a single point. Check: recall 0.50 and specificity ≈ 0.81 give (0.50 + 0.81) / 2 ≈ **0.655**, which is exactly the ROC-AUC.

**Q2. Will the CV ROC-AUC beat KNN (0.779)? How does it compare with LR?**
- *My prediction:* better than KNN, because CV averages several decision trees; unsure about LR.
- *Result:* worse than both KNN and LR (0.655) ❌
- *Corrected reason:* CV trains **5 separate trees**, one per fold, and averages their **scores**. Each tree is still judged alone, so CV *measures* the model and doesn't improve it. Averaging the *predictions* of many trees is a different technique: **bagging**, which is the **Random Forest** in step 5.

**Q3. Fit time vs SVM (2.3 s)?**
- *My prediction:* faster, because the tree doesn't compare pairs of customers.
- *Result:* faster, 0.19 s ✅
- *Reason:* correct. The tree sorts each feature and tries split points, so its cost grows roughly with n log n per feature, with no pairwise customer comparisons.

**Q4. A tree's rules are easy to read. Why is a fully grown tree still useless to show a retention manager?**
- *My prediction:* a huge tree is harder to explain.
- *Result:* partly ✅
- *Full answer:* it is **too big to read** (hundreds of leaves, rules many levels deep) **and not trustworthy**: many leaves hold only 1–2 customers, so those rules are memorised noise that won't hold for new customers. Size check: `pipe_dt.fit(X_train, y_train).named_steps["model"].get_n_leaves()`.

**Takeaway:** the default tree is the worst model so far (ROC-AUC 0.655, gap 0.345). It fails L6 badly, and it was expected to. It shows why a tree must be limited (`max_depth`, `min_samples_leaf`) or averaged (Random Forest).

### Decision tree (Limited)

Two settings stop the tree early:

* `max_depth`: the maximum number of questions from the root down to a leaf.
* `min_samples_leaf`: a leaf must hold at least this many customers, so no single-customer rules.

**Your task:** pick values yourself and add a row such as `"Decision Tree (depth=?, leaf=?)"`. Before you choose, think about these:

1. What `min_samples_leaf` would make a leaf's churn rate meaningful? Is 1% of 5634 customers per leaf too strict or too loose?
2. How deep can a tree go before a manager can no longer read it?

In [20]:
from sklearn.tree import DecisionTreeClassifier
pipe_dt_ltd = build_pipeline(DecisionTreeClassifier(random_state=config.RANDOM_STATE, max_depth=10, min_samples_leaf=50), model_type="tree")
results_cv.append(run_cv(pipe_dt_ltd, "Decision Tree (depth = 10, leaf = 50)", X=X_train, y=y_train))


In [21]:
model_compare_df = pd.DataFrame(results_cv)
model_compare_df 

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.125040
1,KNN,0.779182,0.018299,0.521791,0.032812,0.577296,0.520401,0.547279,0.121479,0.109721
2,SVM (RBF),0.795707,0.017552,0.636447,0.021620,0.656116,0.503679,0.569373,0.070347,2.263394
3,Decision Tree (default),0.654996,0.015929,0.376298,0.016837,0.483949,0.500334,0.491915,0.344985,0.192541
4,"Decision Tree (depth = 10, leaf = 50)",0.831225,0.012703,0.635536,0.027891,0.646169,0.494314,0.558334,0.031916,0.092308


#### Decision Tree (depth=10, leaf=50): choosing the limits, predictions vs result

| | LR | KNN (k=5) | SVM (RBF) | Tree (default) | Tree (depth=10, leaf=50) |
|---|---|---|---|---|---|
| CV ROC-AUC | 0.845 | 0.779 | 0.796 | 0.655 | **0.831** |
| CV PR-AUC | 0.660 | 0.522 | 0.636 | 0.376 | **0.636** |
| gap (train − CV ROC-AUC) | 0.003 | 0.121 | 0.070 | 0.345 | **0.032** |
| fit_time (s) | 0.13 | 0.11 | 2.26 | 0.19 | **0.09** |

**Task 1. What `min_samples_leaf` makes a leaf's churn rate meaningful?**
- *My answer:* 1% is good; 1% of 5634 is about 56 customers per leaf. How do I infer this?
- *How to infer it:* a leaf's churn rate is an **estimate** from its customers. Its uncertainty is the standard error √(p(1−p)/n). With the base churn rate p ≈ 0.27:

| customers in leaf | ± 95% range on the leaf's churn rate |
|---|---|
| 10 | ± 27 points |
| 20 | ± 19 points |
| 50 | ± 12 points |
| 100 | ± 9 points |

With ~50 customers you can trust the difference between a "10% churn" leaf and a "50% churn" leaf, but not between 25% and 30%. So ~1% is a sensible starting point. Two refinements:
- Inside CV each training fold has about 4,507 rows (4/5 of 5634), so 50 is about 1.1% of a fold.
- A leaf of 50 customers is also a segment big enough for the business to act on.

The final value comes from CV in step 9. This calculation only tells you where to start the search.

**Task 2. How deep can a tree go before a manager can no longer read it?**
- *My answer:* beyond about 50 splits it becomes hard to read. How do I decide this?
- *Correction:* **depth** isn't the number of splits. Depth is the number of questions on *one* path from the root to a leaf, so depth 10 means a rule with up to 10 conditions. The number of leaves can grow to 2^depth (1,024 at depth 10). With leaf=50, a fold can hold at most about 90 leaves, which is still far too many to read.
- *How to decide:* readability and accuracy pick **two different trees**:
  - **For the manager:** depth 3, at most 8 leaves. It fits on a slide, and each rule has at most 3 conditions ("month-to-month and fiber and tenure ≤ 12 → high churn").
  - **For scoring:** the depth that CV picks (step 9). Nobody needs to read that tree.
- *Check what the tree actually became* (with leaf=50 it may stop before depth 10): `m = pipe_dt_ltd.fit(X_train, y_train).named_steps["model"]`, then `m.get_depth()` and `m.get_n_leaves()`.

**Q1. Will the gap drop below 0.05 (L6)?**
- *My prediction:* yes, because less noise is learnt and there is less overfitting.
- *Result:* yes, 0.032 ✅ (it passes L6)
- *Reason:* correct. Leaves of at least 50 customers can't memorise single customers.

**Q2. Will the CV ROC-AUC beat KNN and SVM? Will it beat LR?**
- *My prediction:* comparable to KNN and SVM (~0.78), below LR.
- *Result:* 0.831, clearly **above** KNN and SVM ❌, and below LR (−0.014) ✅
- *Corrected reason:*
  - **Above KNN and SVM:** the tree *chooses* which features to split on (Contract and tenure first), while the distance models weigh every column equally.
  - **Below LR:** the tree describes smooth effects (tenure, MonthlyCharges) as **steps**, and the churn signal here is mostly additive, which is exactly what LR models well.

**Q3. Why can a *smaller* tree score higher on CV than the full one?**
- *My prediction:* fewer leaves, so less noise is learnt and there is less overfitting.
- *Result:* ✅, with one more reason:
  1. **Less noise** (bias–variance): a little more bias, much less variance, so lower error on new customers.
  2. **It can rank again:** leaves of 50 customers give probabilities like 0.12 or 0.47 instead of only 0 or 1, so customers can be sorted by risk, and ROC-AUC measures ranking.

**Takeaway:** the limited tree is the best non-LR model so far. It is the fastest (0.09 s) and passes L6, but it's still below LR (0.831 vs 0.845). The final `max_depth` / `min_samples_leaf` / `ccp_alpha` come from GridSearchCV in step 9.